# Rota de um pedido

Mostra as paradas de um pedido em um layout e a distância, o tempo e o ganho de cada estratégia. Para ver outro caso basta trocar `PEDIDO` e `LAYOUT`.

In [1]:
%run grafo.ipynb
%run rotas.ipynb

from itertools import combinations

PEDIDO, LAYOUT = 532393, "L0"

Os 100 pedidos de avaliação:

In [2]:
pedidos.assign(n_secoes=pedidos["secoes"].str.len())[["id_teste", "order_id", "n_itens", "n_secoes"]]

,id_teste,order_id,n_itens,n_secoes
0,1,21461,12,3
1,2,40098,8,7
2,3,44675,28,16
3,4,66497,18,10
4,5,116525,6,5
...,...,...,...,...
95,96,3317375,11,9
96,97,3333851,6,3
97,98,3351108,7,6
98,99,3371845,5,4


In [3]:
pedido = pedidos.set_index("order_id").loc[PEDIDO]
slots = slots_do_pedido(pedido["secoes"], carregar_layout(LAYOUT), loja)
nomes = nomes_das_secoes()

print(f"Pedido {PEDIDO}, layout {LAYOUT}: {pedido['n_itens']} itens, {len(slots)} seções, "
      f"{len(set(loja.corredor[slots]))} corredores")
print("Paradas (corredor, posição) na ordem do carrinho:")
for secao, slot in zip(pedido["secoes"], slots):
    print(f"  ({loja.corredor[slot]:>2},{loja.posicao[slot]})  {nomes[secao]}")

Pedido 532393, layout L0: 10 itens, 8 seções, 6 corredores
Paradas (corredor, posição) na ordem do carrinho:
  ( 8,4)  fresh fruits
  ( 3,8)  cream
  (15,6)  packaged vegetables fruits
  (16,5)  water seltzer sparkling water
  ( 3,3)  bread
  ( 9,3)  chips pretzels
  ( 3,4)  fruit vegetable snacks
  ( 2,1)  ice cream ice


In [4]:
linhas = []
for estrategia in ESTRATEGIAS:
    ordem, metros = executar(estrategia, slots, distancias, loja)
    linhas.append({"estrategia": estrategia, "distancia_m": metros,
                   "tempo_min": tempo_coleta(metros, pedido["n_itens"]) / 60,
                   "ordem": " -> ".join(f"({loja.corredor[s]},{loja.posicao[s]})" for s in ordem)})

rotas = pd.DataFrame(linhas).set_index("estrategia")
carrinho = rotas.loc["ordem_carrinho", "distancia_m"]
rotas["ganho_pct"] = 100 * (carrinho - rotas["distancia_m"]) / carrinho

empates = [f"{a} = {b}" for a, b in combinations(ESTRATEGIAS, 2)
           if abs(rotas.loc[a, "distancia_m"] - rotas.loc[b, "distancia_m"]) <= 1e-9]
print("custoMin:", "exata (Held-Karp)" if usa_held_karp(slots) else "heurística (2-opt)")
print("Estratégias equivalentes:", ", ".join(empates) if empates else "nenhuma")
rotas.style.format({"distancia_m": "{:.1f}", "tempo_min": "{:.1f}", "ganho_pct": "{:.0f}%"})

custoMin: exata (Held-Karp)
Estratégias equivalentes: vizinho = custoMin


,distancia_m,tempo_min,ordem,ganho_pct
estrategia,,,,
ordem_carrinho,364.0,10.6,"(8,4) -> (3,8) -> (15,6) -> (16,5) -> (3,3) -> (9,3) -> (3,4) -> (2,1)",0%
corredores,174.0,7.4,"(2,1) -> (3,8) -> (3,4) -> (3,3) -> (8,4) -> (9,3) -> (15,6) -> (16,5)",52%
vizinho,159.0,7.2,"(2,1) -> (3,3) -> (3,4) -> (3,8) -> (8,4) -> (9,3) -> (15,6) -> (16,5)",56%
custoMin,159.0,7.2,"(2,1) -> (3,3) -> (3,4) -> (3,8) -> (8,4) -> (9,3) -> (15,6) -> (16,5)",56%
